# 🚀 Data Augmentation para Facturas - VERSIÓN ALEATORIA + CROP INTELIGENTE

## ✨ NUEVO: Sistema de CROP + DESPLAZAMIENTO ALEATORIO

Este notebook expande tu dataset de facturas con **calidad vectorial 100% preservada**, **orden aleatorio** y **variaciones ÚNICAS**.

### 🎯 ¿Qué hace?
- ✅ **FASE 1 - AUTO-CROP**: Elimina automáticamente márgenes blancos del PDF
- ✅ **FASE 2 - DESPLAZAMIENTO ALEATORIO**: Shifts únicos de 45-60px en direcciones aleatorias
- ✅ **Genera de 1 a 16 variaciones por factura** (TÚ ELIGES el número)
- ✅ **Mantiene texto vectorial** (SIN rasterizar)
- ✅ **Procesa en orden aleatorio** (evita sesgos de entrenamiento)
- ✅ Organiza dataset automáticamente

### 🔥 ¿Por qué desplazamientos ALEATORIOS?

**VENTAJAS:**
- Cada PDF augmentado es ÚNICO (no hay duplicados)
- Direcciones aleatorias (±X, ±Y) generan máxima variabilidad
- Rango moderado (45-60px) evita distorsión extrema
- Perfecto balance entre variabilidad y realismo

**CÓMO FUNCIONA:**
- Cada transformación genera shifts aleatorios en X e Y
- Valores: random(45, 60) píxeles por eje
- Direcciones: aleatorias (positivas o negativas)
- **RESULTADO**: Cada factura augmentada es ÚNICA

### 📊 Ejemplos con 100 facturas:
- **NUM_TRANSFORMATIONS = 5**  → 600 archivos (1 original + 5 variaciones ÚNICAS)
- **NUM_TRANSFORMATIONS = 10** → 1,100 archivos (1 original + 10 variaciones ÚNICAS)
- **NUM_TRANSFORMATIONS = 16** → 1,700 archivos (1 original + 16 variaciones ÚNICAS)

### 🎛️ Sistema de generación aleatoria:
- Cada transformación usa valores ÚNICOS
- No hay categorías fijas ("small", "medium", "large")
- Rango: 45-60 píxeles por eje
- Direcciones: aleatorias (±X, ±Y)
- **Resultado**: Máxima variabilidad sin repeticiones

### 🔥 Mejoras implementadas:
1. **🆕 Desplazamientos ALEATORIOS** (45-60px con direcciones aleatorias)
2. **Sistema CROP + DESPLAZAMIENTO** (variaciones visualmente notorias)
3. **Manipulación directa de PDF** (sin rasterizar)
4. **DPI 300** (estándar profesional para OCR)
5. **Variaciones configurables** (1-16, TÚ ELIGES)
6. **Procesamiento aleatorio** (evita sesgos en training)
7. **Modo rápido** (3-5x más rápido con SSD local)
8. **Sharpening profesional** para imágenes PNG/JPG
9. **Validación automática** de calidad

## 📦 Paso 1: Instalación y Setup

In [ ]:
# Instalar dependencias (incluyendo pypdf y opencv para manipulación directa)
!pip install -q Pillow pdf2image numpy pypdf>=3.0.0 opencv-python>=4.8.0
!apt-get install -qq poppler-utils

print("✅ Dependencias instaladas")
print("   • Pillow: Procesamiento de imágenes")
print("   • pdf2image: Conversión PDF a imagen (solo para PNG/JPG)")
print("   • pypdf: Manipulación directa de PDF (SIN rasterizar)")
print("   • opencv-python: Detección automática de contenido (auto-crop)")
print("   • poppler-utils: Herramientas PDF")

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive montado en /content/drive")

In [ ]:
# Clonar el repositorio actualizado
import os

# Eliminar si ya existe
if os.path.exists('modificador-de-espacios-de-facturas'):
    !rm -rf modificador-de-espacios-de-facturas
    print("🗑️ Versión anterior eliminada")

# Clonar branch con mejoras
!git clone -b claude/fix-pdf-image-generation-011CUyt3zRDhvutDkiZdkXVn https://github.com/GynoRomeroPrado/modificador-de-espacios-de-facturas.git
%cd modificador-de-espacios-de-facturas

print("\n✅ Repositorio clonado (branch con mejoras)")
print("   Branch: claude/fix-pdf-image-generation-011CUyt3zRDhvutDkiZdkXVn")

## ⚙️ Paso 2: Configuración

**IMPORTANTE:** Modifica estas rutas según tu estructura de Google Drive

In [ ]:
# ========================================
# CONFIGURA ESTAS RUTAS Y PARÁMETROS
# ========================================

# Directorio con tus facturas originales (PDF/imagen + JSON)
INPUT_DIR = "/content/drive/MyDrive/Facturas"

# Directorio donde guardar el dataset augmentado
OUTPUT_DIR = "/content/drive/MyDrive/Facturas_Procesadas"

# DPI para convertir PDFs a imágenes (solo si necesitas PNG/JPG)
# NOTA: Si tus facturas son PDFs, se procesarán directamente SIN rasterizar
DPI = 300  # Estándar profesional para OCR

# ========================================
# ⚙️ NÚMERO DE VARIACIONES (CONFIGURABLE)
# ========================================
# Elige cuántas variaciones generar por factura (1-16)
#
# SISTEMA DE DESPLAZAMIENTOS ALEATORIOS:
#   Cada variación recibe desplazamientos ÚNICOS y ALEATORIOS
#   • Rango: 45-60 píxeles por eje (X e Y)
#   • Direcciones: Aleatorias (puede ser +X/-X y +Y/-Y)
#   • Resultado: Cada PDF es ÚNICO (sin repeticiones)
#
# ¿Cómo funciona?
#   1. El sistema genera valores aleatorios entre 45-60px
#   2. Aplica direcciones aleatorias (±X, ±Y)
#   3. Cada transformación es DIFERENTE
#
# Ejemplos:
#   Variación 1: shift_x = +52px, shift_y = -48px (derecha-arriba)
#   Variación 2: shift_x = -59px, shift_y = +45px (izquierda-abajo)
#   Variación 3: shift_x = +47px, shift_y = +56px (derecha-abajo)
#   ... cada una es ÚNICA
#
# Recomendaciones:
#   • 5 variaciones: Buena variabilidad con dataset moderado
#   • 10 variaciones: Muy buena variabilidad (RECOMENDADO)
#   • 16 variaciones: Máxima variabilidad (datasets grandes)
#
# Ejemplo con 100 facturas:
#   NUM_TRANSFORMATIONS = 5  → 600 archivos totales (1 original + 5 únicas)
#   NUM_TRANSFORMATIONS = 10 → 1,100 archivos totales (1 original + 10 únicas)
#   NUM_TRANSFORMATIONS = 16 → 1,700 archivos totales (1 original + 16 únicas)

NUM_TRANSFORMATIONS = 10  # ← CAMBIA ESTE NÚMERO (1-16)

# ========================================

print(f"📁 Input:  {INPUT_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"🔧 DPI:    {DPI} (solo para rasterización de imágenes)")
print(f"🔢 Variaciones por factura: {NUM_TRANSFORMATIONS}")
print(f"🎲 Sistema de desplazamientos: ALEATORIO (45-60px)")
print(f"   Cada variación es ÚNICA con:")
print(f"   • shift_x: random(45, 60) * random(±1)")
print(f"   • shift_y: random(45, 60) * random(±1)")
print(f"\n💡 NOTA: Los PDFs se procesarán directamente (sin rasterizar)")
print(f"   para mantener texto vectorial 100% seleccionable")

## 🔍 Paso 3: Verificar Facturas de Entrada

In [ ]:
import os
from pathlib import Path

# Verificar que el directorio existe
if not os.path.exists(INPUT_DIR):
    print(f"❌ ERROR: El directorio {INPUT_DIR} no existe")
    print("\nPor favor:")
    print("1. Verifica que la ruta sea correcta")
    print("2. Asegúrate de haber montado Google Drive")
else:
    # Buscar archivos
    input_path = Path(INPUT_DIR)
    pdf_files = list(input_path.glob('*.pdf'))
    image_files = list(input_path.glob('*.jpg')) + list(input_path.glob('*.jpeg')) + list(input_path.glob('*.png'))
    json_files = list(input_path.glob('*.json'))
    
    print(f"✅ Directorio encontrado: {INPUT_DIR}")
    print(f"\n📊 Contenido:")
    print(f"  • Archivos PDF:       {len(pdf_files)}")
    print(f"  • Archivos imagen:    {len(image_files)}")
    print(f"  • Archivos JSON:      {len(json_files)}")
    
    # Verificar pares
    all_files = pdf_files + image_files
    pairs = 0
    pdf_pairs = 0
    image_pairs = 0
    
    for file in all_files:
        json_file = file.with_suffix('.json')
        if json_file.exists():
            pairs += 1
            if file.suffix.lower() == '.pdf':
                pdf_pairs += 1
            else:
                image_pairs += 1
    
    print(f"  • Pares completos:    {pairs}")
    print(f"    - PDFs vectoriales: {pdf_pairs} (MEJOR - sin rasterizar)")
    print(f"    - Imágenes:         {image_pairs} (con mejoras OCR)")
    
    if pairs == 0:
        print("\n⚠️  ADVERTENCIA: No se encontraron pares completos (archivo + JSON)")
        print("   Cada factura debe tener su JSON correspondiente")
    else:
        print(f"\n✅ Listo para procesar {pairs} facturas")
        print(f"   Resultado esperado con {NUM_TRANSFORMATIONS} variaciones:")
        print(f"   • Originales:   {pairs}")
        print(f"   • Augmentadas:  {pairs * NUM_TRANSFORMATIONS}")
        print(f"   • TOTAL:        {pairs * (1 + NUM_TRANSFORMATIONS)} archivos")
        
        if pdf_pairs > 0:
            print(f"\n🎯 EXCELENTE: {pdf_pairs} PDFs vectoriales")
            print(f"   → Se procesarán SIN rasterizar")
            print(f"   → Texto 100% seleccionable preservado")
            print(f"   → Calidad original 100%")
            print(f"   → Orden de procesamiento aleatorio (evita sesgos)")

## 📋 Paso 4: Ver Transformaciones Configuradas

In [ ]:
import sys
sys.path.append('/content/modificador-de-espacios-de-facturas/src')

from augmentation import AugmentationConfig

# Crear configuración con el número de transformaciones especificado
config = AugmentationConfig(num_transformations=NUM_TRANSFORMATIONS)

print("🔧 CONFIGURACIÓN DE TRANSFORMACIONES")
print("=" * 60)
print(f"\n🎲 Sistema de Desplazamientos ALEATORIOS:")
print(f"  • Rango mínimo:  {config.SHIFT_MIN}px")
print(f"  • Rango máximo:  {config.SHIFT_MAX}px")
print(f"  • Direcciones:   Aleatorias (±X, ±Y)")
print(f"\nNúmero de transformaciones configurado: {config.num_transformations}")
print(f"\n💡 Ejemplo de transformaciones generadas (valores ALEATORIOS):")
print("   Cada vez que proceses, los valores serán DIFERENTES:")

# Generar un ejemplo de transformaciones
example_transforms = config.generate_random_shifts()
for transform in example_transforms:
    direction = ""
    if transform['shift_x'] > 0:
        direction += "→ "
    else:
        direction += "← "
    if transform['shift_y'] > 0:
        direction += "↓"
    else:
        direction += "↑"
    
    print(f"  {transform['index']:2d}. shift_x={transform['shift_x']:+3d}px, shift_y={transform['shift_y']:+3d}px  {direction}")

print(f"\n💡 Cada factura generará:")
print(f"   1 original + {config.num_transformations} variaciones ÚNICAS = {1 + config.num_transformations} archivos")
print(f"\n🎯 Ventajas del sistema aleatorio:")
print(f"   ✅ Cada PDF es ÚNICO (no hay repeticiones)")
print(f"   ✅ Máxima variabilidad sin patrones predecibles")
print(f"   ✅ Mejor generalización en modelos de ML")
print("=" * 60)

## 🚀 Paso 5A: Procesar Dataset (MODO NORMAL)

Este paso genera todas las variaciones augmentadas guardando directamente en Google Drive.

**NOTA:**
- PDFs se procesarán directamente (SIN rasterizar)
- Imágenes se procesarán con mejoras OCR
- Los archivos se guardan directamente en Drive (más lento pero no requiere copia final)

In [ ]:
from main import InvoiceDatasetAugmenter

# Crear augmenter con número de transformaciones configurable
augmenter = InvoiceDatasetAugmenter(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    dpi=DPI,
    num_transformations=NUM_TRANSFORMATIONS  # ← Configurable
)

# Procesar dataset
print("🚀 INICIANDO PROCESAMIENTO (MODO NORMAL)...\n")
stats = augmenter.process_dataset()
print("\n✅ PROCESAMIENTO COMPLETADO")

## ⚡ Paso 5B: Procesar Dataset (MODO RÁPIDO - RECOMENDADO)

**🚀 3-5x MÁS RÁPIDO que el modo normal**

Este modo procesa todo en el disco local de Colab (muy rápido) y luego copia a Drive al final.

**Ventajas:**
- ⚡ 3-5x más rápido (especialmente con muchas facturas)
- 💾 Usa el SSD local de Colab (muy rápido)
- 📦 Copia comprimida final a Drive

**Desventajas:**
- 🔄 Requiere copia final a Drive (incluida automáticamente)
- 💾 Requiere espacio en disco local (~2x tamaño del dataset)

**IMPORTANTE:** Si tienes más de 1,000 facturas, usa ESTE modo.

In [ ]:
import shutil
import time
from main import InvoiceDatasetAugmenter

print("⚡ MODO RÁPIDO ACTIVADO")
print("=" * 60)

# Usar directorio local temporal (SSD rápido de Colab)
LOCAL_OUTPUT = "/content/facturas_temp"
print(f"\n📁 Procesando en disco local: {LOCAL_OUTPUT}")
print(f"   (Mucho más rápido que Google Drive)")

# Crear augmenter con salida local y número de transformaciones configurable
augmenter = InvoiceDatasetAugmenter(
    input_dir=INPUT_DIR,
    output_dir=LOCAL_OUTPUT,
    dpi=DPI,
    num_transformations=NUM_TRANSFORMATIONS  # ← Configurable
)

# Medir tiempo
start_time = time.time()

# Procesar dataset
print("\n🚀 INICIANDO PROCESAMIENTO (MODO RÁPIDO)...\n")
stats = augmenter.process_dataset()

processing_time = time.time() - start_time
print(f"\n✅ PROCESAMIENTO COMPLETADO en {processing_time:.1f} segundos")

# Copiar a Google Drive
print(f"\n📦 Copiando resultados a Google Drive...")
print(f"   Origen: {LOCAL_OUTPUT}")
print(f"   Destino: {OUTPUT_DIR}")

copy_start = time.time()

# Copiar archivos
if os.path.exists(LOCAL_OUTPUT):
    # Crear directorio de destino si no existe
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Copiar todo el directorio
    # Primero eliminar destino si existe
    if os.path.exists(OUTPUT_DIR):
        print(f"   🗑️ Limpiando destino existente...")
        shutil.rmtree(OUTPUT_DIR)
    
    # Copiar
    shutil.copytree(LOCAL_OUTPUT, OUTPUT_DIR)
    
    copy_time = time.time() - copy_start
    total_time = time.time() - start_time
    
    print(f"\n✅ COPIA COMPLETADA en {copy_time:.1f} segundos")
    print(f"\n⏱️ TIEMPO TOTAL: {total_time:.1f} segundos")
    print(f"   • Procesamiento: {processing_time:.1f}s")
    print(f"   • Copia a Drive: {copy_time:.1f}s")
    
    # Limpiar directorio local
    print(f"\n🧹 Limpiando archivos temporales locales...")
    shutil.rmtree(LOCAL_OUTPUT)
    print(f"   ✅ Limpieza completada")
    
    print(f"\n🎯 Dataset guardado en: {OUTPUT_DIR}")
else:
    print(f"\n❌ Error: No se encontró el directorio local {LOCAL_OUTPUT}")

print("\n" + "=" * 60)

## ⚙️ Comparación de Rendimiento

### ⏱️ Tiempos Estimados (4,300 facturas):

| Modo | Tiempo Aprox | Velocidad | Recomendado para |
|------|-------------|-----------|------------------|
| **Modo Normal** (5A) | 2-3 horas | 1x | <100 facturas |
| **Modo Rápido** (5B) | 30-45 min | 3-5x | >100 facturas |

### 🎯 ¿Qué modo usar?

**Usa MODO NORMAL (Paso 5A) si:**
- Tienes pocas facturas (<100)
- Quieres ver resultados en Drive inmediatamente
- No te importa esperar más tiempo

**Usa MODO RÁPIDO (Paso 5B) si:**
- Tienes muchas facturas (>100)
- Quieres ahorrar tiempo significativo
- Tienes espacio en el disco local de Colab

### 💡 ¿Por qué es más rápido el Modo Rápido?

```
Google Drive I/O:  🐢 Lento (red)
Disco local Colab: 🚀 Rápido (SSD)

Modo Normal:  [Leer Drive] → [Procesar] → [Escribir Drive lento] ❌
Modo Rápido:  [Leer Drive] → [Procesar] → [Escribir SSD rápido] → [Copiar Drive] ✅

Ganancia: 3-5x más rápido
```

### ⚠️ Limitaciones de GPU/TPU

**GPU/TPU NO aceleran este proceso** porque:
- Las operaciones son principalmente I/O (lectura/escritura de archivos)
- pypdf usa transformaciones matemáticas simples (no paralelizables en GPU)
- Pillow no tiene soporte GPU nativo

**Las GPU/TPU son útiles para:**
- Entrenar modelos de ML (después de generar el dataset)
- Procesamiento de imágenes con TensorFlow/PyTorch
- Inferencia de modelos

**Este proceso está optimizado para:**
- Maximizar velocidad de I/O (disco local vs Drive)
- Procesamiento CPU eficiente
- Transformaciones vectoriales directas

## ✅ Paso 6: Verificar Resultados

In [ ]:
import json

# Leer reporte
report_path = os.path.join(OUTPUT_DIR, 'dataset_report.json')

if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    print("📊 REPORTE DEL PROCESO")
    print("=" * 60)
    print(f"\n📅 Fecha: {report['timestamp']}")
    print(f"\n📁 Directorios:")
    print(f"  • Input:  {report['input_directory']}")
    print(f"  • Output: {report['output_directory']}")
    print(f"\n📊 Estadísticas:")
    print(f"  • Facturas originales:  {report['statistics']['original_invoices']}")
    print(f"  • Facturas augmentadas: {report['statistics']['augmented_invoices']}")
    print(f"  • Total de facturas:    {report['statistics']['total_invoices']}")
    print(f"\n🔧 Configuración:")
    print(f"  • Transformaciones:     {report['augmentation_config']['total_transformations']}")
    print(f"  • Rangos (px):          {report['augmentation_config']['shift_levels']}")
    
    if report['statistics']['errors']:
        print(f"\n⚠️  Errores: {len(report['statistics']['errors'])}")
        for error in report['statistics']['errors']:
            print(f"    - {error}")
    else:
        print("\n✅ Sin errores")
    
    expansion = report['statistics']['total_invoices'] / report['statistics']['original_invoices']
    print(f"\n🚀 Dataset expandido {expansion:.1f}x")
    print("=" * 60)
else:
    print("❌ No se encontró el reporte")

## 👀 Paso 7: Explorar Resultados

In [ ]:
# Listar archivos generados
organized_dir = os.path.join(OUTPUT_DIR, 'organized')
augmented_dir = os.path.join(OUTPUT_DIR, 'augmented')

print("📁 ESTRUCTURA DE ARCHIVOS GENERADOS\n")
print(f"{OUTPUT_DIR}/")

if os.path.exists(organized_dir):
    organized_files = list(Path(organized_dir).glob('*'))
    print(f"├── organized/ ({len(organized_files)} archivos)")
    print(f"│   ├── Facturas originales renombradas")
    print(f"│   ├── Formato: factura_XXXX.pdf/json")
    for f in sorted(organized_files)[:3]:
        print(f"│   ├── {f.name}")
    if len(organized_files) > 3:
        print(f"│   └── ... y {len(organized_files) - 3} más")

print(f"│")

if os.path.exists(augmented_dir):
    augmented_files = list(Path(augmented_dir).glob('*'))
    print(f"├── augmented/ ({len(augmented_files)} archivos)")
    print(f"│   ├── Variaciones augmentadas (10 por factura)")
    print(f"│   ├── Formato: factura_XXXX_aug_XX_direccion.pdf/json")
    for f in sorted(augmented_files)[:10]:
        print(f"│   ├── {f.name}")
    if len(augmented_files) > 10:
        print(f"│   └── ... y {len(augmented_files) - 10} más")

print(f"│")
print(f"└── dataset_report.json (reporte completo)")

## 🔍 Paso 8: Verificar Calidad Vectorial (Solo PDFs)

In [ ]:
from pypdf import PdfReader

# Buscar primer PDF augmentado
augmented_dir = os.path.join(OUTPUT_DIR, 'augmented')
augmented_pdfs = sorted(Path(augmented_dir).glob('*.pdf'))

if augmented_pdfs:
    test_pdf = augmented_pdfs[0]
    
    print(f"🔬 VERIFICANDO CALIDAD VECTORIAL\n")
    print(f"Archivo de prueba: {test_pdf.name}")
    print(f"=" * 60)
    
    try:
        reader = PdfReader(test_pdf)
        page = reader.pages[0]
        text = page.extract_text()
        
        print(f"\n✅ Texto vectorial extraído con éxito")
        print(f"\nCaracteres extraídos: {len(text)}")
        print(f"\nMuestra de texto:")
        print(f"{'─' * 60}")
        print(text[:300] + "..." if len(text) > 300 else text)
        print(f"{'─' * 60}")
        
        if len(text) > 0:
            print(f"\n🎯 RESULTADO: Texto vectorial PRESERVADO")
            print(f"   ✅ Texto 100% seleccionable")
            print(f"   ✅ Calidad original mantenida")
            print(f"   ✅ Perfecto para OCR")
        else:
            print(f"\n⚠️  ADVERTENCIA: No se extrajo texto")
            print(f"   Posiblemente sea una imagen rasterizada")
    except Exception as e:
        print(f"\n❌ Error al leer PDF: {e}")
else:
    print("ℹ️  No se encontraron PDFs augmentados para verificar")
    print("   (Esto es normal si solo procesaste imágenes PNG/JPG)")

## 📊 Paso 9: Estadísticas de Tamaño de Archivos

In [ ]:
import os
from pathlib import Path

def get_folder_size(folder_path):
    """Calcula el tamaño total de una carpeta en MB"""
    total_size = 0
    for file in Path(folder_path).rglob('*'):
        if file.is_file():
            total_size += file.stat().st_size
    return total_size / (1024 * 1024)  # Convertir a MB

organized_dir = os.path.join(OUTPUT_DIR, 'organized')
augmented_dir = os.path.join(OUTPUT_DIR, 'augmented')

print("💾 ESTADÍSTICAS DE ALMACENAMIENTO\n")

if os.path.exists(organized_dir):
    size_organized = get_folder_size(organized_dir)
    count_organized = len(list(Path(organized_dir).glob('*')))
    print(f"📂 organized/")
    print(f"   • Archivos: {count_organized}")
    print(f"   • Tamaño:   {size_organized:.2f} MB")
    print(f"   • Promedio: {size_organized/count_organized:.2f} MB/archivo\n")

if os.path.exists(augmented_dir):
    size_augmented = get_folder_size(augmented_dir)
    count_augmented = len(list(Path(augmented_dir).glob('*')))
    print(f"📂 augmented/")
    print(f"   • Archivos: {count_augmented}")
    print(f"   • Tamaño:   {size_augmented:.2f} MB")
    print(f"   • Promedio: {size_augmented/count_augmented:.2f} MB/archivo\n")

if os.path.exists(organized_dir) and os.path.exists(augmented_dir):
    total_size = size_organized + size_augmented
    total_count = count_organized + count_augmented
    print(f"📊 TOTAL")
    print(f"   • Archivos: {total_count}")
    print(f"   • Tamaño:   {total_size:.2f} MB")
    print(f"   • Promedio: {total_size/total_count:.2f} MB/archivo")

## 🎯 Próximos Pasos

### ✅ Dataset Generado Exitosamente

Tu dataset ha sido expandido y está listo para usar:

**📁 Ubicación:** `{OUTPUT_DIR}`

**📊 Estructura:**
```
Facturas_Procesadas/
├── organized/          # Facturas originales
│   ├── factura_0001.pdf
│   ├── factura_0001.json
│   └── ...
├── augmented/          # Variaciones ÚNICAS por factura
│   ├── factura_0001_aug_01.pdf (desplazamiento aleatorio único)
│   ├── factura_0001_aug_02.pdf (desplazamiento aleatorio único)
│   ├── factura_0001_aug_03.pdf (desplazamiento aleatorio único)
│   ├── factura_0001_aug_04.pdf (desplazamiento aleatorio único)
│   ├── factura_0001_aug_05.pdf (desplazamiento aleatorio único)
│   ├── ...
│   └── ... (+ JSONs correspondientes)
└── dataset_report.json # Reporte completo
```

### 🚀 Usa este dataset para:

1. **Entrenar modelos de detección**
   - YOLOv8, Faster R-CNN, etc.
   - Variaciones aleatorias = mejor generalización

2. **Generar bounding boxes**
   - Más datos únicos = anotación más robusta
   - Cubre posiciones impredecibles de campos

3. **Mejorar sistemas OCR**
   - PDFs: Texto vectorial 100% seleccionable
   - Imágenes: Mejoras profesionales aplicadas

### 💡 Ventajas de este dataset:

- ✅ **PDFs vectoriales**: Calidad 100% preservada
- ✅ **Variaciones ÚNICAS**: Cada PDF es diferente (sistema aleatorio)
- ✅ **Organizado**: Estructura clara y JSONs actualizados
- ✅ **Dataset balanceado**: 1 original + N variaciones únicas
- ✅ **OCR Ready**: DPI 300 + mejoras profesionales
- ✅ **Sin sesgos**: Procesamiento en orden aleatorio

### 📝 Sistema de Transformaciones Aleatorias:

**Rango de desplazamientos:**
- **Mínimo**: 45 píxeles
- **Máximo**: 60 píxeles
- **Direcciones**: Aleatorias (±X, ±Y)

**Ejemplo de variaciones generadas:**
```
factura_0001_aug_01.pdf → shift_x: +52px, shift_y: -48px
factura_0001_aug_02.pdf → shift_x: -59px, shift_y: +45px
factura_0001_aug_03.pdf → shift_x: +47px, shift_y: +56px
factura_0001_aug_04.pdf → shift_x: -51px, shift_y: -60px
...
```

**Características:**
- Cada transformación usa valores ÚNICOS
- No hay repeticiones ni patrones predecibles
- Máxima variabilidad para entrenamiento

### 📋 Notas importantes:

1. **PDFs procesados directamente**:
   - NO se rasterizaron
   - Texto 100% seleccionable
   - Tamaño de archivo pequeño

2. **Imágenes PNG/JPG**:
   - Mejoras OCR aplicadas
   - DPI 300
   - Sharpening profesional

3. **JSONs actualizados**:
   - Campo `filename` actualizado
   - Metadata de augmentation incluida:
     - `shift_x`: desplazamiento en X (píxeles)
     - `shift_y`: desplazamiento en Y (píxeles)
     - `shift_range`: "45-60px"
     - `augmentation_index`: número de variación
   - Campo `is_augmented`: true para variaciones

4. **Procesamiento aleatorio**:
   - Las facturas se procesan en orden aleatorio
   - Los desplazamientos son aleatorios
   - Esto evita sesgos durante el entrenamiento
   - Cada ejecución genera valores diferentes